# V6: audit first, bounded detector experiment

V5 Version 5 remains the user-confirmed best official Public PQ **0.32**.
This notebook has no measured solar validation or official score yet.
It embeds repository modules rather than downloading moving GitHub main.
Default execution performs synthetic readiness and artifact inventory, **no training**.
No V5 checkpoint or submission is overwritten. Before training, inspect the current
competition rules and official self-evaluation notebook, archive the latter and
set its path below. It reports public-reference all-pairs PQ and one-to-one local PQ separately; neither is a leaderboard result.
V5 forensics require its original fitting/validation manifests and exported
per-strategy RLE predictions; a new random split is invalid for an old checkpoint.

Repository code MIT; torchvision BSD. V5's Apache reference remains attributed
in the unchanged V5 notebook and third_party/NOTICE.md. No third-party source
notebook or competition data is redistributed here.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
WORK = Path('/kaggle/working/v6') if Path('/kaggle/working').exists() else Path.cwd()/'v6-work'
WORK.mkdir(parents=True, exist_ok=True)
# Keep installed compatible torch/torchvision together; do not replace CUDA wheels blindly.
if Path('/kaggle/working').exists():
    subprocess.run([sys.executable,'-m','pip','install','numpy==1.26.4',
                    'opencv-python-headless==4.10.0.84','scipy==1.13.1','pycocotools==2.0.8'],check=True)
import torch, torchvision
from torchvision.ops import nms
assert nms(torch.tensor([[0.,0.,1.,1.]]),torch.tensor([1.]),.5).numel()==1
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
(WORK/'environment.txt').write_text(freeze)
print('torch',torch.__version__,'torchvision',torchvision.__version__,'GPUs',torch.cuda.device_count())


In [ ]:
sources = {'solar/__init__.py': '', 'solar/data.py': '"""MAGFiLO reader. Splits are grouped by acquisition DAY, not annotation ID.\n\nMultiple annotators see an identical image. Never place that observation\nin both training and validation, even via separate annotation IDs.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nimport cv2\nimport numpy as np\n\nSIZE = 2048\nEXT = {\'.jpg\', \'.jpeg\', \'.png\'}\n\n\ndef discover(root: str | Path) -> tuple[Path, Path, Path]:\n    """Find images/annotations under an unzipped data directory or /kaggle/input."""\n    root = Path(root)\n    found = list(root.rglob(\'MAGFiLO_1.0_Annotations_kaggle2026_train.json\'))\n    if len(found) != 1:\n        raise FileNotFoundError(f\'Expected one training JSON under {root}; found {len(found)}. Attach competition data first.\')\n    ann = found[0]\n    train = ann.parent / \'train_images\'\n    base = ann.parent.parent\n    test = base / \'test\' / \'test_images\'\n    if not train.is_dir() or not test.is_dir():\n        raise FileNotFoundError(f\'Expected {train} and {test}.\')\n    return ann, train, test\n\n\ndef load_records(root: str | Path) -> tuple[list[dict], Path, Path]:\n    ann, train_dir, test_dir = discover(root)\n    with ann.open(encoding=\'utf-8\') as f:\n        payload = json.load(f)\n    by_image = defaultdict(list)\n    for a in payload[\'annotations\']:\n        by_image[str(a[\'image_id\'])].append(a)\n    indexed = {p.stem: p for p in train_dir.rglob(\'*\') if p.suffix.lower() in EXT}\n    records = []\n    for img in payload[\'images\']:\n        stem = Path(img[\'file_name\']).stem\n        if stem not in indexed:\n            raise FileNotFoundError(f\'Missing training image {stem} in {train_dir}\')\n        records.append({\'image_id\': str(img[\'id\']), \'stem\': stem,\n                        \'path\': str(indexed[stem]), \'ann\': by_image[str(img[\'id\'])]})\n    if not records:\n        raise ValueError(\'Empty MAGFiLO training annotations.\')\n    return records, train_dir, test_dir\n\n\ndef split_records(records: list[dict], val_frac: float = .16, seed: int = 42) -> tuple[list[dict], list[dict]]:\n    """Hold out whole observation DATES; all re-annotations share a split."""\n    if not 0 < val_frac < .5:\n        raise ValueError(\'val_frac must be between 0 and 0.5\')\n    days = sorted({r[\'stem\'][:8] for r in records})\n    if len(days) < 2:\n        raise ValueError(\'Need at least two observation days for honest validation.\')\n    rng = np.random.default_rng(seed)\n    rng.shuffle(days)\n    count = min(len(days) - 1, max(1, round(len(days) * val_frac)))\n    val_days = set(days[:count])\n    train = [r for r in records if r[\'stem\'][:8] not in val_days]\n    val = [r for r in records if r[\'stem\'][:8] in val_days]\n    assert {x[\'stem\'] for x in train}.isdisjoint({x[\'stem\'] for x in val})\n    return train, val\n\n\ndef read_gray(path: str | Path) -> np.ndarray:\n    arr = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n    if arr is None:\n        raise ValueError(f\'Cannot read image: {path}\')\n    return arr\n\n\ndef polygons_to_instances(annotations: list[dict], shape=(SIZE, SIZE)) -> np.ndarray:\n    """Render COCO polygon targets; use one annotator\'s labels per sample.\n\n    All source annotations use a single polygon per filament, but we also\n    handle multiple polygons defensively. Label 0 is the background.\n    """\n    canvas = np.zeros(shape, dtype=np.int32)\n    for idx, ann in enumerate(annotations, start=1):\n        seg = ann.get(\'segmentation\') or []\n        if isinstance(seg, dict):\n            raise ValueError(\'RLE training labels not expected; polygon required.\')\n        for polygon in seg:\n            pts = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)\n            if len(pts) < 3:\n                continue\n            pts = np.rint(pts).astype(np.int32)\n            cv2.fillPoly(canvas, [pts], idx)\n    return canvas\n\n\ndef feature_channels(image: np.ndarray) -> np.ndarray:\n    """Two channels, fixed image-wide preprocessing; input shape H,W,2."""\n    image = image.astype(np.float32) / 255.0\n    # Contrast against a broad smooth background; dark filaments go positive.\n    background = cv2.GaussianBlur(image, (0, 0), sigmaX=23.0)\n    dark = np.clip((background - image) * 5.0 + .5, 0, 1)\n    return np.stack([image, dark], axis=-1).astype(np.float32)\n\n\ndef derive_targets(instance: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    fg = (instance > 0).astype(np.float32)\n    # Boundary along adjacent instance borders (where separation matters).\n    boundary = np.zeros_like(fg)\n    for dy, dx in [(0, 1), (1, 0)]:\n        a = instance[:instance.shape[0]-dy or None, :instance.shape[1]-dx or None]\n        b = instance[dy:, dx:]\n        mismatch = (a != b) & (a > 0) & (b > 0)\n        boundary[:boundary.shape[0]-dy or None, :boundary.shape[1]-dx or None] = np.maximum(\n            boundary[:boundary.shape[0]-dy or None, :boundary.shape[1]-dx or None], mismatch)\n    # Thin exterior contours also sharpen localization for small filaments.\n    border = cv2.morphologyEx(fg, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))\n    return fg, np.maximum(boundary, border * .5).astype(np.float32)\n\n\nclass TileDataset:\n    """On-the-fly full-resolution training patches, positive-heavy sampling.\n\n    No pre-cropping of images is needed; avoids a huge intermediary dataset.\n    """\n    def __init__(self, records, tile=512, samples_per_epoch=1200, seed=13, training=True):\n        self.records = records\n        self.tile = tile\n        self.samples = samples_per_epoch\n        self.seed = seed\n        self.training = training\n        self.epoch = 0\n        self.annotators = defaultdict(list)\n        for r in records:\n            self.annotators[r[\'stem\']].append(r)\n        self.unique = sorted(self.annotators)\n\n    def __len__(self):\n        return self.samples\n\n    def set_epoch(self, epoch: int):\n        self.epoch = int(epoch) if self.training else 0\n\n    def __getitem__(self, index):\n        import torch\n        # Independent RNG even with multiple data-loader workers.\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, index, self.epoch]))\n        stem = self.unique[int(rng.integers(len(self.unique)))]\n        variants = self.annotators[stem]\n        rec = variants[int(rng.integers(len(variants)))]\n        im = read_gray(rec[\'path\'])\n        inst = polygons_to_instances(rec[\'ann\'], im.shape)\n        h, w = im.shape\n        t = self.tile\n        if h < t or w < t:\n            raise ValueError(f\'Image {rec["path"]} smaller than tile {t}\')\n        if rng.random() < .78 and inst.max() > 0:\n            ys, xs = np.nonzero(inst)\n            k = int(rng.integers(len(ys)))\n            y = int(np.clip(int(ys[k]) - rng.integers(t//4, 3*t//4), 0, h-t))\n            x = int(np.clip(int(xs[k]) - rng.integers(t//4, 3*t//4), 0, w-t))\n        else:\n            y, x = int(rng.integers(h-t+1)), int(rng.integers(w-t+1))\n        # Image-wide correction before cropping is essential for train/test parity.\n        feat = feature_channels(im)[y:y+t, x:x+t]\n        inst = inst[y:y+t, x:x+t]\n        for _ in range(int(rng.integers(4))):\n            feat, inst = np.rot90(feat), np.rot90(inst)\n        if rng.random() < .5:\n            feat, inst = feat[:, ::-1], inst[:, ::-1]\n        if rng.random() < .5:\n            feat, inst = feat[::-1], inst[::-1]\n        feat = np.ascontiguousarray(feat)\n        # Mild photometric augmentation only on the intensity channel.\n        if self.training:\n            scale = float(rng.uniform(.90, 1.10))\n            shift = float(rng.uniform(-.025, .025))\n            feat[..., 0] = np.clip(feat[..., 0] * scale + shift, 0, 1)\n            # A global affine intensity transform also scales the background\n            # subtraction channel around its neutral value of 0.5.\n            feat[..., 1] = np.clip(.5 + (feat[..., 1] - .5) * scale, 0, 1)\n        fg, boundary = derive_targets(inst)\n        return (torch.from_numpy(feat.transpose(2, 0, 1).copy()),\n                torch.from_numpy(np.stack([fg, boundary], axis=0)))\n', 'solar/v6_audit.py': '"""Overlapping-instance forensic tools. All reported PQ values are LOCAL proxies.\n\nIndependent binary masks deliberately avoid label-map overwrite of overlapping\nannotations. This implementation is not asserted equivalent to Kaggle\'s scorer.\n"""\nfrom __future__ import annotations\nimport csv\nimport json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom pycocotools import mask as coco\n\n\ndef _masks(value):\n    value = np.asarray(value)\n    if value.ndim != 3 or not np.isin(value, [0,1]).all():\n        raise ValueError(\'Expected binary masks with shape [N,H,W]\')\n    if len(value) and (value.sum(axis=(1,2)) == 0).any():\n        raise ValueError(\'Empty instances are invalid; use [0,H,W] for no detections\')\n    return value.astype(bool, copy=False)\n\n\ndef pairwise(gt, pred):\n    gt, pred = _masks(gt), _masks(pred)\n    if gt.shape[1:] != pred.shape[1:]:\n        raise ValueError(\'Mask dimensions differ\')\n    ga = gt.sum(axis=(1,2)).astype(float)\n    pa = pred.sum(axis=(1,2)).astype(float)\n    # Avoid N*M*H*W temporaries and uint8 matrix multiplication overflow.\n    intersection = np.zeros((len(gt),len(pred)),float)\n    for i,g in enumerate(gt):\n        for j,p in enumerate(pred):\n            intersection[i,j] = np.count_nonzero(g & p)\n    iou = intersection / np.maximum(ga[:,None]+pa[None,:]-intersection,1)\n    return intersection, ga, pa, iou\n\n\ndef measure(gt, pred):\n    intersection,ga,pa,iou = pairwise(gt,pred)\n    # Prioritize valid matches before IoU sum for overlapping annotation masks.\n    valid = iou > .5\n    rows,cols = linear_sum_assignment(-(valid*(min(iou.shape,default=0)+1)+iou))\n    matches = [(int(r),int(c)) for r,c in zip(rows,cols) if valid[r,c]]\n    values = [float(iou[r,c]) for r,c in matches]\n    tp,fp,fn = len(values), len(pa)-len(values), len(ga)-len(values)\n    denominator = tp+.5*(fp+fn)\n    total = sum(values)\n    reference_tp=int(valid.sum())\n    reference_fp=int(np.count_nonzero(~valid.any(axis=0)))\n    reference_fn=int(np.count_nonzero(~valid.any(axis=1)))\n    reference_sum=float(iou[valid].sum())\n    reference_denominator=reference_tp+.5*(reference_fp+reference_fn)\n    return {\n        \'pq_reference\':reference_sum/reference_denominator if reference_denominator else 0.,\n        \'reference_tp\':reference_tp,\'reference_fp\':reference_fp,\'reference_fn\':reference_fn,\n        \'reference_sum_iou\':reference_sum,\n        \'pq_local\':total/denominator if denominator else 1.,\n        \'sq\':total/tp if tp else (1. if not denominator else 0.),\n        \'rq\':tp/denominator if denominator else 1.,\n        \'tp\':tp,\'fp\':fp,\'fn\':fn,\'sum_iou\':total,\n        \'precision\':tp/(tp+fp) if tp+fp else (1. if not fn else 0.),\n        \'recall\':tp/(tp+fn) if tp+fn else 1.,\n        \'matched_iou\':values,\n        \'matched_dice\':[float(2*intersection[r,c]/(ga[r]+pa[c])) for r,c in matches],\n        \'likely_splits\':int(np.count_nonzero((intersection/np.maximum(ga[:,None],1) >= .1).sum(1)>=2)),\n        \'likely_merges\':int(np.count_nonzero((intersection/np.maximum(pa[None,:],1) >= .1).sum(0)>=2)),\n        \'unmatched_gt\':[i for i in range(len(ga)) if i not in {r for r,_ in matches}],\n        \'unmatched_pred\':[i for i in range(len(pa)) if i not in {c for _,c in matches}],\n    }\n\n\ndef partition(records, seed=42):\n    """60/20/20 date partition for NEW runs; never reuse for already trained V5.\n\n    Dates must be eight leading YYYYMMDD characters. V5 audit requires its\n    original training manifest, rather than a newly invented holdout.\n    """\n    import datetime\n    days = sorted({r[\'stem\'][:8] for r in records})\n    for day in days:\n        datetime.datetime.strptime(day,\'%Y%m%d\')\n    if len(days) < 5:\n        raise ValueError(\'Need at least five observation days for three-way evaluation\')\n    np.random.default_rng(seed).shuffle(days)\n    n = max(1,round(.2*len(days)))\n    tuning,confirmation = set(days[:n]),set(days[n:2*n])\n    return tuple([r for r in records if (r[\'stem\'][:8] in group)]\n                 for group in (set(days)-tuning-confirmation,tuning,confirmation))\n\n\ndef bootstrap(observations, seed=42, n=2000, metric="pq_local"):\n    """Macro-observation PQ; average annotators first, bootstrap observations."""\n    if not observations or n < 1 or any(not rows for rows in observations.values()):\n        raise ValueError(\'Need nonempty observation groups and positive replicates\')\n    scores = np.array([np.mean([r[metric] for r in rows]) for rows in observations.values()])\n    rng = np.random.default_rng(seed)\n    samples = [float(rng.choice(scores,len(scores),replace=True).mean()) for _ in range(n)]\n    low,high = np.quantile(samples,[.025,.975])\n    return {\'estimate\':float(scores.mean()),\'low\':float(low),\'high\':float(high),\n            \'metric\':metric,\'unit\':\'observation\',\'aggregation\':\'macro observation; annotators averaged\',\n            \'observations\':len(scores),\'replicates\':n,\'seed\':seed}\n\n\ndef export_submission(predictions, image_stems, dest):\n    """Validate the full inventory, round-trip compressed COCO RLE, then export."""\n    stems = list(image_stems)\n    if len(set(stems)) != len(stems) or set(predictions) != set(stems):\n        raise ValueError(\'Duplicate image stems or incomplete/extra image predictions\')\n    rows=[]\n    for stem in stems:\n        masks = _masks(predictions[stem])\n        if masks.shape[1:] != (2048,2048):\n            raise ValueError(\'Competition masks must be 2048 by 2048\')\n        for i,mask in enumerate(masks,1):\n            encoded = coco.encode(np.asfortranarray(mask.astype(np.uint8)))\n            if not np.array_equal(coco.decode(encoded),mask):\n                raise ValueError(\'COCO RLE round trip failed\')\n            rows.append({\'filament_id\':f\'{stem}_{i}\',\'segmentation_rle\':encoded[\'counts\'].decode(\'ascii\')})\n    if not rows or len({r[\'filament_id\'] for r in rows}) != len(rows):\n        raise ValueError(\'Empty submission or duplicate instance IDs\')\n    dest=Path(dest); dest.parent.mkdir(parents=True,exist_ok=True)\n    with dest.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=[\'filament_id\',\'segmentation_rle\'])\n        writer.writeheader(); writer.writerows(rows)\n    dest.with_suffix(\'.inventory.json\').write_text(json.dumps({k:len(v) for k,v in predictions.items()},indent=2))\n\n\ndef overlay(image,gt,pred,path):\n    """Two panels with individual contours: green GT; red predictions."""\n    gt,pred = _masks(gt),_masks(pred)\n    if image.shape != gt.shape[1:] or image.shape != pred.shape[1:]:\n        raise ValueError(\'Image-mask shape mismatch\')\n    panels=[]\n    for masks,color in ((gt,(0,255,0)),(pred,(0,0,255))):\n        panel=cv2.cvtColor(image,cv2.COLOR_GRAY2BGR)\n        for i,mask in enumerate(masks,1):\n            contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n            cv2.drawContours(panel,contours,-1,color,1)\n            y,x=np.argwhere(mask)[0]; cv2.putText(panel,str(i),(int(x),int(y)),cv2.FONT_HERSHEY_SIMPLEX,.4,color,1)\n        panels.append(panel)\n    Path(path).parent.mkdir(parents=True,exist_ok=True)\n    if not cv2.imwrite(str(path),np.concatenate(panels,axis=1)):\n        raise OSError(f\'Cannot save {path}\')\n', 'solar/v6_instances.py': '"""Optional V6 postprocessing ablations; no threshold is claimed validated.\n\nKeep binary masks separate through evaluation. Resolving overlaps is optional,\nchanges their area and must be compared on validation before use.\n"""\nimport numpy as np\nfrom .v6_audit import _masks\n\n\ndef _scores(masks,scores):\n    masks = _masks(masks)\n    scores = np.asarray(scores,dtype=float)\n    if scores.shape != (len(masks),) or not np.isfinite(scores).all():\n        raise ValueError(\'Need one finite score per instance\')\n    return masks,scores\n\n\ndef mask_nms(masks,scores,iou_threshold=.5):\n    """Stable mask IoU NMS; never union neighboring instances."""\n    masks,scores = _scores(masks,scores)\n    if not 0 < iou_threshold < 1:\n        raise ValueError(\'NMS IoU threshold must lie in (0,1)\')\n    keep=[]\n    for idx in np.argsort(-scores,kind=\'stable\'):\n        duplicate=False\n        for previous in keep:\n            shared = np.count_nonzero(masks[idx]&masks[previous])\n            union = np.count_nonzero(masks[idx]|masks[previous])\n            if shared/union > iou_threshold:\n                duplicate=True; break\n        if not duplicate: keep.append(int(idx))\n    return keep\n\n\ndef resolve_overlap(masks,scores):\n    """Assign shared pixels to the highest-confidence instance, no dilation."""\n    masks,scores = _scores(masks,scores)\n    occupied=np.zeros(masks.shape[1:],bool)\n    result=np.zeros_like(masks)\n    for idx in np.argsort(-scores,kind=\'stable\'):\n        result[idx] = masks[idx]&~occupied\n        occupied |= masks[idx]\n    return result\n\n\ndef restore_tile(masks,origin,shape):\n    """Place N native-resolution tile masks into full-image coordinates."""\n    masks = _masks(masks)\n    y,x = origin; h,w = masks.shape[1:]\n    if y < 0 or x < 0 or y+h > shape[0] or x+w > shape[1]:\n        raise ValueError(\'Tile exceeds image bounds\')\n    output=np.zeros((len(masks),*shape),bool)\n    output[:,y:y+h,x:x+w] = masks\n    return output\n', 'solar/v6_forensics.py': '"""Offline V5 diagnostic runner, independent of private Kaggle authentication.\n\nPrediction JSON: {strategy: {physical_image_stem: [{size:[H,W], counts:str}]}}.\nTraining manifest: JSON list of every physical stem used for fitting the model\nOR its refiner. A new random split cannot establish that V5 never saw an image.\n"""\nfrom __future__ import annotations\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom pycocotools import mask as coco\nfrom .v6_audit import measure, bootstrap, overlay\n\n\ndef run_audit(ground_truth,predictions,out):\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    if not ground_truth or not predictions:\n        raise ValueError(\'Need ground truth and prediction strategies\')\n    report={}\n    for strategy,by_image in predictions.items():\n        if set(by_image) != set(ground_truth):\n            raise ValueError(f\'{strategy}: missing/extra physical observation predictions\')\n        observations={}\n        for stem,variants in ground_truth.items():\n            if not variants: raise ValueError(\'Observation has no annotation variants\')\n            shape=variants[0].shape[1:]\n            masks=[]\n            for rle in by_image[stem]:\n                if tuple(rle[\'size\']) != shape: raise ValueError(\'Prediction dimensions differ\')\n                masks.append(coco.decode({\'size\':rle[\'size\'],\'counts\':rle[\'counts\'].encode(\'ascii\')}))\n            pred=np.stack(masks).astype(bool) if masks else np.zeros((0,*shape),bool)\n            observations[stem]=[measure(gt,pred) for gt in variants]\n        rows=[row for variants in observations.values() for row in variants]\n        tp,fp,fn=(sum(r[k] for r in rows) for k in (\'tp\',\'fp\',\'fn\'))\n        total=sum(r[\'sum_iou\'] for r in rows); denominator=tp+.5*(fp+fn)\n        reference_tp,reference_fp,reference_fn=(sum(r[k] for r in rows) for k in (\'reference_tp\',\'reference_fp\',\'reference_fn\'))\n        reference_sum=sum(r[\'reference_sum_iou\'] for r in rows)\n        reference_denom=reference_tp+.5*(reference_fp+reference_fn)\n        report[strategy]={\n            \'public_reference_micro\':{\'pq_reference\':reference_sum/reference_denom if reference_denom else 0.,\n                                      \'tp\':reference_tp,\'fp\':reference_fp,\'fn\':reference_fn},\n            \'public_reference_macro_ci\':bootstrap(observations,metric=\'pq_reference\'),\n            \'micro\':{\'pq_local\':total/denominator if denominator else 1.,\n                     \'sq\':total/tp if tp else (1. if not denominator else 0.),\n                     \'rq\':tp/denominator if denominator else 1.,\'tp\':tp,\'fp\':fp,\'fn\':fn},\n            \'macro_observation_ci\':bootstrap(observations),\n            \'observations\':observations,\n            \'metric_status\':\'pq_reference reproduces inspected public helper semantics; pq_local uses one-to-one matching; neither is a leaderboard result\',\n        }\n    (out/\'comparison.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    return report\n\n\ndef load_holdout(data,train_stems,validation_stems):\n    from .data import load_records,read_gray\n    records,_,_=load_records(data)\n    fitting=set(train_stems); selected=set(validation_stems)\n    if not fitting or not selected or fitting&selected:\n        raise ValueError(\'Need nonempty, disjoint original fitting and validation manifests\')\n    if {s[:8] for s in fitting}&{s[:8] for s in selected}:\n        raise ValueError(\'Training and holdout dates overlap\')\n    known={r[\'stem\'] for r in records}\n    if not (fitting|selected)<=known: raise ValueError(\'Unknown stems in split manifest\')\n    truth={s:[] for s in sorted(selected)}; paths={}\n    for record in records:\n        stem=record[\'stem\']\n        if stem not in selected: continue\n        image=read_gray(record[\'path\']); h,w=image.shape\n        masks=[]\n        for annotation in record[\'ann\']:\n            seg=annotation.get(\'segmentation\')\n            if not isinstance(seg,list): raise ValueError(\'Expected official train polygons\')\n            rles=coco.frPyObjects(seg,h,w)\n            if rles:\n                mask=coco.decode(coco.merge(rles)).astype(bool)\n                if mask.any(): masks.append(mask)\n        truth[stem].append(np.stack(masks) if masks else np.zeros((0,h,w),bool))\n        paths[stem]=record[\'path\']\n    return truth,paths\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data\',required=True)\n    parser.add_argument(\'--predictions\',type=Path,required=True)\n    parser.add_argument(\'--fitting-stems\',type=Path,required=True)\n    parser.add_argument(\'--validation-stems\',type=Path,required=True)\n    parser.add_argument(\'--out\',type=Path,required=True)\n    args=parser.parse_args()\n    truth,paths=load_holdout(args.data,json.loads(args.fitting_stems.read_text()),json.loads(args.validation_stems.read_text()))\n    predictions=json.loads(args.predictions.read_text())\n    report=run_audit(truth,predictions,args.out)\n    from .data import read_gray\n    for strategy,by_image in predictions.items():\n        # Numeric filenames keep strategy/observation names out of output paths.\n        folder=args.out/f\'strategy-{list(predictions).index(strategy)}\'\n        index={}\n        for number,(stem,variants) in enumerate(truth.items()):\n            shape=variants[0].shape[1:]\n            pred=[coco.decode({\'size\':r[\'size\'],\'counts\':r[\'counts\'].encode()}) for r in by_image[stem]]\n            pred=np.stack(pred) if pred else np.zeros((0,*shape),bool)\n            for annotator,gt in enumerate(variants):\n                name=f\'{number:04d}-{annotator:02d}.png\'\n                overlay(read_gray(paths[stem]),gt,pred,folder/name)\n                index[name]={\'observation\':stem,\'annotation_variant\':annotator}\n        (folder/\'index.json\').write_text(json.dumps(index,indent=2))\n    print(json.dumps({k:v[\'micro\'] for k,v in report.items()},indent=2))\n\n\nif __name__==\'__main__\': main()\n', 'solar/v6_detector.py': '"""V6 torchvision detector adapter, separate from and compatible with no V5 files.\n\nNew checkpoints use this module\'s explicit format. Never guess the structure of\nV5 checkpoints or load untrusted pickles. No pretrained downloads in CPU tests.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport torch\nfrom torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\n\n\ndef build_detector(pretrained=True,min_size=1024,max_size=1024):\n    if min_size < 32 or max_size < min_size:\n        raise ValueError(\'Require 32 <= min_size <= max_size\')\n    model=maskrcnn_resnet50_fpn_v2(\n        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None,\n        weights_backbone=None,min_size=min_size,max_size=max_size)\n    model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,2)\n    model.roi_heads.mask_predictor=MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,2)\n    return model\n\n\ndef save_checkpoint(model,path,config):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    torch.save({\'format\':\'solar-v6-detector-1\',\'state_dict\':model.state_dict(),\'config\':json.loads(json.dumps(config))},path)\n\n\ndef load_checkpoint(path,device=\'cpu\'):\n    checkpoint=torch.load(path,map_location=\'cpu\',weights_only=True)\n    if checkpoint.get(\'format\') != \'solar-v6-detector-1\':\n        raise ValueError(\'Not a V6 detector checkpoint; V5 files require their original loader\')\n    config=checkpoint[\'config\']\n    model=build_detector(False,config[\'min_size\'],config[\'max_size\'])\n    model.load_state_dict(checkpoint[\'state_dict\'],strict=True)\n    return model.to(device).eval(),config\n\n\n@torch.inference_mode()\ndef infer(model,image,score_threshold=.5,mask_threshold=.5):\n    if image.ndim != 2 or image.dtype != np.uint8:\n        raise ValueError(\'Expected uint8 grayscale image\')\n    if not 0 <= score_threshold <= 1 or not 0 < mask_threshold < 1:\n        raise ValueError(\'Invalid confidence/mask threshold\')\n    device=next(model.parameters()).device\n    tensor=torch.from_numpy(image.copy()).float().div(255).unsqueeze(0).expand(3,-1,-1).to(device)\n    model.eval()\n    prediction=model([tensor])[0]\n    keep=prediction[\'scores\']>=score_threshold\n    masks=(prediction[\'masks\'][keep,0]>=mask_threshold).cpu().numpy()\n    scores=prediction[\'scores\'][keep].cpu().numpy()\n    nonempty=masks.any(axis=(1,2))\n    return masks[nonempty],scores[nonempty]\n', 'solar/v6_train.py': '"""Bounded V6 detector experiment. Not a validated replacement for V5.\n\nStarts only with explicit --run-training and an archived official evaluator.\nCurrent local metric remains a proxy until equivalence is independently tested.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom pycocotools import mask as coco\nfrom .data import load_records,read_gray\nfrom .v6_audit import partition,measure,bootstrap\nfrom .v6_detector import build_detector,infer,save_checkpoint,load_checkpoint\n\n\nclass DetectionDataset:\n    def __init__(self,records): self.records=records\n    def __len__(self): return len(self.records)\n    def __getitem__(self,index):\n        record=self.records[index]\n        image=read_gray(record[\'path\']); h,w=image.shape\n        masks=[]; boxes=[]\n        for annotation in record[\'ann\']:\n            polygons=annotation[\'segmentation\']\n            if not isinstance(polygons,list): raise ValueError(\'Expected train polygons\')\n            rles=coco.frPyObjects(polygons,h,w)\n            if not rles: continue\n            mask=coco.decode(coco.merge(rles))\n            yy,xx=np.nonzero(mask)\n            if not len(xx): continue\n            masks.append(mask); boxes.append([xx.min(),yy.min(),xx.max()+1,yy.max()+1])\n        tensor=torch.from_numpy(image.copy()).float().div(255).unsqueeze(0).expand(3,-1,-1)\n        target={\'boxes\':torch.tensor(boxes,dtype=torch.float32).reshape(-1,4),\n                \'labels\':torch.ones(len(masks),dtype=torch.int64),\n                \'masks\':torch.tensor(np.stack(masks) if masks else np.zeros((0,h,w)),dtype=torch.uint8),\n                \'image_id\':torch.tensor(index)}\n        return tensor,target\n\n\ndef evaluate(model,records,score_threshold=.5):\n    groups={}; started=time.perf_counter()\n    grouped={}\n    for record in records:\n        grouped.setdefault(record[\'stem\'],[]).append(record)\n    # Release full-resolution predictions after each observation. Never cache\n    # the whole validation split (hundreds of MB per observation).\n    for stem,variants in grouped.items():\n        pred=infer(model,read_gray(variants[0][\'path\']),score_threshold)[0]\n        dataset=DetectionDataset(variants)\n        groups[stem]=[measure(dataset[i][1][\'masks\'].numpy(),pred) for i in range(len(dataset))]\n    result=bootstrap(groups,metric="pq_reference")\n    rows=[row for variants in groups.values() for row in variants]\n    tp,fp,fn=(sum(row[k] for row in rows) for k in (\'reference_tp\',\'reference_fp\',\'reference_fn\'))\n    denominator=tp+.5*(fp+fn)\n    result[\'reference_micro_pq\']=sum(row[\'reference_sum_iou\'] for row in rows)/denominator if denominator else 0.\n    result[\'evaluation_seconds_per_observation\']=(time.perf_counter()-started)/len(groups)\n    result[\'by_observation\']=groups\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data\',required=True); p.add_argument(\'--out\',type=Path,required=True)\n    p.add_argument(\'--official-evaluation\',type=Path,required=True)\n    p.add_argument(\'--run-training\',action=\'store_true\')\n    p.add_argument(\'--epochs\',type=int,default=2); p.add_argument(\'--max-steps\',type=int,default=100)\n    p.add_argument(\'--resolution\',type=int,default=1024); p.add_argument(\'--seed\',type=int,default=42)\n    args=p.parse_args()\n    if not args.run_training: p.error(\'Training requires explicit --run-training\')\n    if not args.official_evaluation.is_file(): p.error(\'Archive and inspect official evaluator first\')\n    if args.epochs<1 or args.max_steps<1: p.error(\'Epoch and per-epoch step budget must be positive\')\n    if not torch.cuda.is_available(): p.error(\'GPU required for this bounded research run\')\n    if args.out.exists() and any(args.out.iterdir()): p.error(\'Use a new empty output directory; preserve previous runs\')\n    args.out.mkdir(parents=True,exist_ok=True)\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)\n    torch.backends.cudnn.benchmark=False\n    records,_,_=load_records(args.data)\n    train,tune,confirm=partition(records,args.seed)\n    config={\'seed\':args.seed,\'min_size\':args.resolution,\'max_size\':args.resolution,\n            \'epochs\':args.epochs,\'max_steps_per_epoch\':args.max_steps,\'learning_rate\':.0001,\n            \'metric_status\':\'micro-aggregated public-reference PQ for selection, macro observation bootstrap CI; not an official score\',\n            \'official_evaluation_sha256\':hashlib.sha256(args.official_evaluation.read_bytes()).hexdigest(),\n            \'torch\':str(torch.__version__),\'gpu\':torch.cuda.get_device_name(0),\n            \'device_count_visible\':torch.cuda.device_count(),\'devices_used\':1}\n    (args.out/\'config.json\').write_text(json.dumps(config,indent=2))\n    (args.out/\'split.json\').write_text(json.dumps({name:[{\'stem\':r[\'stem\'],\'image_id\':r[\'image_id\']} for r in part]\n                                                 for name,part in [(\'train\',train),(\'tune\',tune),(\'confirmation\',confirm)]},indent=2))\n    model=build_detector(True,args.resolution,args.resolution).cuda()\n    loader=torch.utils.data.DataLoader(DetectionDataset(train),batch_size=1,shuffle=True,\n                                        collate_fn=lambda batch:tuple(zip(*batch)),num_workers=0)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=config[\'learning_rate\'],weight_decay=.0001)\n    scaler=torch.amp.GradScaler(\'cuda\'); best=-1.; history=[]; start=time.perf_counter()\n    for epoch in range(args.epochs):\n        model.train(); losses=[]\n        for step,(images,targets) in enumerate(loader):\n            if step>=args.max_steps: break\n            images=[im.cuda() for im in images]; targets=[{k:v.cuda() for k,v in t.items()} for t in targets]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16): loss=sum(model(images,targets).values())\n            if not torch.isfinite(loss): raise RuntimeError(\'Non-finite training loss\')\n            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()\n            losses.append(float(loss.detach()))\n        validation=evaluate(model,tune)\n        row={\'epoch\':epoch+1,\'loss\':float(np.mean(losses)),\'tuning_pq_reference\':validation[\'reference_micro_pq\']}\n        history.append(row); print(json.dumps(row),flush=True)\n        if validation[\'reference_micro_pq\']>best:\n            best=validation[\'reference_micro_pq\']; save_checkpoint(model,args.out/\'best.pt\',config)\n        (args.out/\'history.json\').write_text(json.dumps(history,indent=2))\n    model,_=load_checkpoint(args.out/\'best.pt\',\'cuda\')\n    confirmation=evaluate(model,confirm)\n    confirmation[\'training_seconds\']=time.perf_counter()-start\n    confirmation[\'peak_cuda_bytes\']=torch.cuda.max_memory_allocated()\n    (args.out/\'confirmation.json\').write_text(json.dumps(confirmation,indent=2))\n    print(\'Completed bounded experiment, NOT a final full-data model or official submission.\')\n\n\nif __name__==\'__main__\': main()\n'}
source_hashes = {'solar/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'solar/data.py': 'a24a8c82162fee2d8f47ab9ad5b805fa03cf4ef49d78a85aef7477bfcaee094b', 'solar/v6_audit.py': 'd06c6c3f4f41d0ae6e593e9389b98b4187c25861d55229d184c111fdf46064ea', 'solar/v6_instances.py': 'fa05e0a4f10a6bee20ea61481e308af255badd1d42ce8c653a59a4c4997620f9', 'solar/v6_forensics.py': '9db648053f47d9cb1eea997b5ae4b99b7984cb3b3679c5dfeaf00dee65fc39aa', 'solar/v6_detector.py': 'd412f84ce12a08204dc2f5862ea33b7ab030d676a37200100a4b97d9e576c85c', 'solar/v6_train.py': '2d1b669ffe2cc7dd02408a78dc3228b1baad511431ecdcb33e536b63c559f29e'}
PACKAGE = WORK/'package'
for name,code in sources.items():
    path=PACKAGE/name
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.read_text()!=code:
        raise RuntimeError(f'Refusing to overwrite changed module {path}; use a fresh work directory')
    path.write_text(code)
sys.path.insert(0,str(PACKAGE))
(WORK/'source-hashes.json').write_text(json.dumps(source_hashes,indent=2))


In [ ]:
import numpy as np
from solar.v6_audit import measure
from solar.v6_instances import mask_nms
synthetic=np.zeros((2,32,32),bool)
synthetic[0,2:15,2:15]=1; synthetic[1,10:25,10:25]=1
assert measure(synthetic,synthetic)['pq_local']==1
assert mask_nms(np.concatenate([synthetic,synthetic[:1]]),[.9,.8,.7])==[0,1]
# These are software checks, NOT solar validation or official metrics.
print('Synthetic readiness passed; no research accuracy measured.')
INPUT=Path('/kaggle/input')
interesting=[str(p) for p in INPUT.rglob('*') if p.is_file() and p.suffix in {'.pt','.pth','.json','.csv','.ipynb'}] if INPUT.exists() else []
(WORK/'input-inventory.json').write_text(json.dumps(interesting,indent=2))
print('Input artifact count:',len(interesting))


In [ ]:
# Optional V5 audit: set all four paths to real, original artifacts.
DATA = '/kaggle/input'
V5_PREDICTIONS = None  # JSON: {strategy: {stem: [{size:[2048,2048],counts:str}]}}
V5_FITTING_STEMS = None  # JSON list, union of detector AND refiner fitting stems
V5_VALIDATION_STEMS = None  # JSON list of original held-out physical stems
if all([V5_PREDICTIONS,V5_FITTING_STEMS,V5_VALIDATION_STEMS]):
    subprocess.run([sys.executable,'-m','solar.v6_forensics','--data',DATA,
                    '--predictions',V5_PREDICTIONS,'--fitting-stems',V5_FITTING_STEMS,
                    '--validation-stems',V5_VALIDATION_STEMS,'--out',str(WORK/'v5-audit')],
                   cwd=PACKAGE,check=True)
else:
    print('V5 forensics pending original predictions and fitting/validation manifests; no failure-mode claim.')


In [ ]:
RUN_TRAINING = False
OFFICIAL_EVALUATION = None  # archived, inspected official evaluator notebook; never test labels
if RUN_TRAINING:
    assert OFFICIAL_EVALUATION and Path(OFFICIAL_EVALUATION).is_file(), 'Inspect and archive official evaluation first'
    assert torch.cuda.is_available(), 'Enable GPU before explicit bounded run'
    subprocess.run([sys.executable,'-m','solar.v6_train','--data',DATA,
                    '--out',str(WORK/'detector-pilot'),'--official-evaluation',OFFICIAL_EVALUATION,
                    '--run-training','--epochs','2','--max-steps','100','--resolution','1024'],
                   cwd=PACKAGE,check=True)
else:
    print('No GPU training requested; V5 remains best confirmed Public PQ 0.32.')
